# Analisis de Sensibilidad: Optimizacion por Baterias

Evaluacion del impacto de diferentes capacidades de bateria (2.5–30 kWh) sobre el autoconsumo solar, reduccion de excedentes y energia adicional no vendida por la distribuidora.

In [0]:
from pyspark.sql import functions as F

# Ubicacion de las tablas de salida (entorno CNFL)
CATALOGO = "datosami"
ESQUEMA = "datos"

# Resumen global por escenario de bateria
global_df = spark.read.table(f"{CATALOGO}.{ESQUEMA}.impacto_bateria_q3_global")

# Extraer capacidad numerica del nombre del escenario para ordenar
global_sorted = global_df.withColumn(
    "CAPACIDAD_KWH",
    F.regexp_replace(
        F.regexp_extract("ESCENARIO_BATERIA", r"BATERIA_(\d+(?:_\d+)?)", 1),
        "_", "."
    ).cast("double")
).orderBy("CAPACIDAD_KWH")

display(global_sorted.select(
    "CAPACIDAD_KWH",
    "CLIENTES_SIMULADOS",
    F.round("CAPACIDAD_TOTAL_MWH", 2).alias("CAPACIDAD_TOTAL_MWH"),
    F.round("GENERACION_SOLAR_GWH", 2).alias("GEN_SOLAR_GWH"),
    F.round("ENERGIA_CARGADA_GWH", 4).alias("CARGADA_GWH"),
    F.round("ENERGIA_DESCARGADA_GWH", 4).alias("DESCARGADA_GWH"),
    F.round("EXCEDENTE_NETO_GWH", 4).alias("EXCEDENTE_NETO_GWH"),
    F.round("REDUCCION_VENTAS_ADICIONAL_GWH", 4).alias("REDUCCION_ADIC_GWH"),
    F.round("PORCENTAJE_EXCEDENTE_NETO", 2).alias("%_EXCEDENTE_NETO"),
))

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Datos para grafico de sensibilidad
data = global_sorted.select(
    "CAPACIDAD_KWH",
    F.col("EXCEDENTE_NETO_GWH").alias("Excedente Neto (GWh)"),
    F.col("REDUCCION_VENTAS_ADICIONAL_GWH").alias("Reduccion Adicional (GWh)"),
    F.col("PORCENTAJE_EXCEDENTE_NETO").alias("% Excedente Neto"),
).toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Reduccion de ventas y excedente neto vs capacidad
ax1.plot(data["CAPACIDAD_KWH"], data["Reduccion Adicional (GWh)"], "bo-", linewidth=2, markersize=8, label="Reduccion Adicional")
ax1.plot(data["CAPACIDAD_KWH"], data["Excedente Neto (GWh)"], "rs-", linewidth=2, markersize=8, label="Excedente Neto")
ax1.set_xlabel("Capacidad de Bateria (kWh)", fontsize=12)
ax1.set_ylabel("Energia (GWh)", fontsize=12)
ax1.set_title("Sensibilidad: Impacto vs Capacidad de Bateria", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Grafico 2: % Excedente neto restante
ax2.plot(data["CAPACIDAD_KWH"], data["% Excedente Neto"], "g^-", linewidth=2, markersize=10)
ax2.axhline(y=7.3, color="orange", linestyle="--", linewidth=1.5, label="Solo Solar (7.3%)")
ax2.set_xlabel("Capacidad de Bateria (kWh)", fontsize=12)
ax2.set_ylabel("% Excedente Neto", fontsize=12)
ax2.set_title("Reduccion de Excedentes Solares", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Curva de pato comparativa: Solo Solar vs Solar+Bateria
solar_pato = spark.read.table(f"{CATALOGO}.{ESQUEMA}.impacto_solar_q3_curva_pato")
bateria_pato = spark.read.table(f"{CATALOGO}.{ESQUEMA}.impacto_bateria_q3_curva_pato")

# Filtrar para un mes y tipo de dia especifico
mes = 12  # Diciembre (unico mes disponible en la base)
tipo_dia = "laboral"  # TIPO_DIA de ref_calendario: laboral / fin_de_semana (feriado = fin_de_semana)
escenario_bat = "BATERIA_10"

solar_data = solar_pato.filter((F.col("MES") == mes) & (F.col("TIPO_DIA") == tipo_dia)).orderBy("BLOQUE_15M").toPandas()
bat_data = bateria_pato.filter(
    (F.col("MES") == mes) & (F.col("TIPO_DIA") == tipo_dia) & (F.col("ESCENARIO_BATERIA") == escenario_bat)
).orderBy("BLOQUE_15M").toPandas()

fig, ax = plt.subplots(figsize=(14, 6))

x = bat_data["BLOQUE_15M"]
ax.plot(x, solar_data["CURVA_BASE"], "k-", linewidth=2, label="Demanda Base", alpha=0.7)
ax.plot(x, solar_data["GENERACION_SOLAR"], "y-", linewidth=2, label="Generacion Solar", alpha=0.7)
ax.plot(x, solar_data["CURVA_NETA"], "b--", linewidth=2, label="Demanda Neta (Solo Solar)")
ax.plot(x, bat_data["CURVA_NETA"], "r-", linewidth=2, label="Demanda Neta (Solar+Bateria)")
ax.fill_between(x, solar_data["CURVA_NETA"], bat_data["CURVA_NETA"], alpha=0.2, color="green", label="Reduccion por Bateria")

ax.set_xlabel("Bloque de 15 min", fontsize=12)
ax.set_ylabel("Energia (kWh)", fontsize=12)
ax.set_title(f"Curva de Pato: Solo Solar vs Solar+Bateria {escenario_bat} | Mes {mes} {tipo_dia}", fontsize=13)
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 96, 8))

plt.tight_layout()
plt.show()

In [0]:
# Comparacion: Solo Solar vs Solar+Bateria
solar_global = spark.read.table(f"{CATALOGO}.{ESQUEMA}.impacto_solar_q3_global")
bat_global = spark.read.table(f"{CATALOGO}.{ESQUEMA}.impacto_bateria_q3_global")

# Solo Solar: energia no vendida y excedente
solar_row = solar_global.select(
    F.lit("SOLO_SOLAR").alias("ESCENARIO"),
    F.lit(0.0).alias("CAPACIDAD_KWH"),
    F.round(F.col("ENERGIA_NO_VENDIDA_GWH"), 4).alias("NO_VENDIDA_GWH"),
    F.round(F.col("EXCEDENTE_SOLAR_GWH"), 4).alias("EXCEDENTE_GWH"),
    F.round(F.col("PORCENTAJE_REDUCCION_VENTAS"), 2).alias("%_REDUCCION_VENTAS"),
)

# Solar+Bateria: energia no vendida total = solar + adicional por bateria
bat_rows = bat_global.withColumn(
    "CAPACIDAD_KWH",
    F.regexp_replace(
        F.regexp_extract("ESCENARIO_BATERIA", r"BATERIA_(\d+(?:_\d+)?)", 1),
        "_", "."
    ).cast("double")
).select(
    F.col("ESCENARIO_BATERIA").alias("ESCENARIO"),
    F.col("CAPACIDAD_KWH"),
    F.round(F.col("REDUCCION_VENTAS_ADICIONAL_GWH") + F.lit(0.0), 4).alias("NO_VENDIDA_ADIC_GWH"),
    F.round(F.col("EXCEDENTE_NETO_GWH"), 4).alias("EXCEDENTE_NETO_GWH"),
    F.round(F.col("PORCENTAJE_EXCEDENTE_NETO"), 2).alias("%_EXCEDENTE_NETO"),
).orderBy("CAPACIDAD_KWH")

display(bat_rows)

## Hallazgos Clave

1. **Reduccion marginal decreciente**: Cada kWh adicional de bateria aporta menos reduccion de excedentes que el anterior (rendimientos decrecientes)
2. **Bateria de 10 kWh** (mediana): reduce el excedente solar de 7.3% a ~7.1% y aporta ~0.018 GWh de reduccion adicional de ventas
3. **Bateria de 30 kWh** (grande): reduce el excedente a ~6.9% pero con 6x la capacidad de la mediana — el ROI disminuye
4. **Estrategia AUTOCONSUMO_MAX**: la bateria prioriza cargar con excedentes solares y descargar en deficit, maximizando autoconsumo
5. **Eficiencia de ciclo**: round-trip ~0.90 (0.95 carga × 0.95 descarga) — consistente en todos los escenarios